In [11]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path(r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

df = pd.read_csv(data_path)
df.head()

C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [12]:
# import torch
# import shutil
# import os

# save_directory = os.getcwd()

# # Pack the entire directory into a ZIP
# zip_path = shutil.make_archive(
#     base_dir=os.getcwd(),
#     base_name = "rajan1",
#     format="zip",
#     root_dir=r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data"
# )

# print("Saved files:", os.listdir(save_directory))
# print("ZIP location:", zip_path)

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["model"] = le.fit_transform(df["model"])
le.transform(le.classes_)
df.head()

# import pickle
# path = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\deberta_v3_finetuned"
# with open(os.path.join(path, "le.pkl"), "wb") as f:
#     pickle.dump(le, f)

,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",17
1,0041ac86ee93,p073,Data science communities matter because they a...,4
2,005a19083eae,p120,"In the final twilight of time, the last embers...",6
3,009319596f10,p073,Data science communities matter because they t...,14
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",10


### DeBERTa -> Tokenizer 

In [14]:
import os
HF_TOKEN = os.getenv("HF_TOKEN")


In [15]:
from transformers import ( AutoModelForSequenceClassification, AutoTokenizer)

LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\deberta_v3_finetuned"
NUM_LABELS = 18
MODEL_NAME = "microsoft/deberta-v3-small"


tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_DIR
    #    MODEL_NAME, tokends=HF_TOKEN
    )

model = AutoModelForSequenceClassification.from_pretrained(
         LOCAL_DIR
        # MODEL_NAME,
        # num_labels=NUM_LABELS,
        # torch_dtype=torch.float32
    )

# model.save_pretrained(LOCAL_DIR)
# tokenizer.save_pretrained(LOCAL_DIR)
model


c:\Users\ps302\anaconda3\envs\genai\lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Loading weights: 100%|██████████| 106/106 [00:00<00:00, 7999.64it/s]


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-5): 6 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNor

In [16]:
text_df = df.sample(n=5, random_state=42)[["text", "model"]]
text_df

,text,model
1339,"Across geological deep time, Earth’s continent...",12
1222,"In the violent dawn of our solar system, Jupit...",6
1106,Joseph Schumpeter saw capitalism not as a stat...,9
812,The P versus NP problem matters to anyone who ...,14
1231,"In 1859, the Carrington Event sent a massive s...",1


In [17]:
text = text_df["text"].tolist()
len(text)

5

In [18]:
import torch

encoded = tokenizer(
            text, 
            padding=True,
            truncation=True,
            max_length = 256,
            return_tensors = "pt"
        )

model.eval()
with torch.no_grad():
    outputs = model(
        input_ids = encoded["input_ids"],
        attention_mask = encoded["attention_mask"]
    )
    logits = outputs.logits
    
prob = torch.softmax(logits, dim=1)
pred = torch.argmax(prob, dim=1)

print(pred)
print(le.inverse_transform(pred))

tensor([12,  5,  9, 12,  1])
['gpt-5.6-luna' 'gemini-3.1-pro-preview' 'glm-5.3' 'gpt-5.6-luna'
 'claude-opus-4-6']


In [ ]:
# for params in model.deberta.parameters():
#     params.requires_grad = False

In [19]:
# #  switch off the the trainable parameters
for param in model.deberta.parameters():
    param.requires_grad = False

In [20]:
# Total and trainable parameters
total_params = model.num_parameters()
trainable_params = model.num_parameters(only_trainable=True)
print(f"Total: {total_params:,} | Trainable: {trainable_params:,}")

Total: 141,908,754 | Trainable: 604,434


In [21]:
# Returns the total number of parameters
total_params = model.num_parameters()

# Returns only the parameters that require gradients
trainable_params = model.num_parameters(only_trainable=True)

print(f"Total: {total_params:,} | Trainable: {trainable_params:,}")


# Total parameters (including frozen layers)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")

# Trainable parameters only (layers being updated during training)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters: {trainable_params:,}")

# Layer wise parameter
for param in model.parameters():
    print(param.shape)
for name, param in model.named_parameters():
     print(f"Layer: {name} | Shape: {param.shape}")


Total: 141,908,754 | Trainable: 604,434
Total Parameters: 141,908,754
Trainable Parameters: 604,434
torch.Size([128100, 768])
torch.Size([768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([3072, 768])
torch.Size([3072])
torch.Size([768, 3072])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([3072, 768])
torch.Size([3072])
torch.Size([768, 3072])
torch.Size([768])
torch.Size([768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768, 768])
torch.Size([768])
torch.Size([768])
torch.Size([76

In [22]:
text = [
    "The proposed method is efficient.",
    "This answer provides a detailed explanation.",
    "The model generates concise responses."
]

# 	******"padding=True" ka matlab yahan batch ke longest sequence tak padding karna hai.
#    Agar padding="max_length" use karte, toh har sequence ko specified max_length tak pad kiya jata.*****
encoded = tokenizer(
    text,
    truncation=True,
    max_length=128,
    padding=True,
    return_tensors="pt"
)

print(encoded["input_ids"].shape)
print(encoded["attention_mask"].shape)

torch.Size([3, 9])
torch.Size([3, 9])


In [23]:
from torch.onnx._internal.torchscript_exporter.symbolic_opset9 import dim
import torch
outputs = model(**encoded)
logits = outputs.logits
pred = torch.argmax(logits, dim=1)
pred

prob = torch.softmax(logits, dim=1)
prob

predictions = torch.argmax(prob, dim=1)
predictions

tensor([15, 15, 15])

In [34]:
import torch
from torch.utils.data import Dataset

class LLMTextData(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length
        if len(self.texts) != len(self.labels):
            raise ValueError("Texts and labels are not of equal length")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded = self.tokenizer(
            text, 
            truncation=True,
            padding="max_length",
            max_length = self.max_length,
            return_tensors = "pt"
        )

        item = {
            "input_ids": encoded["input_ids"].squeeze(0),     # (1, 10) -> (10,)
            "attention_mask": encoded["attention_mask"].squeeze(0),   # (1, 10) -> (10,)
            "labels": torch.tensor(label, dtype=torch.long)
        }

        return item

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df["model"])

print(train_df.shape, test_df.shape)

train_datasets = LLMTextData(
    texts=train_df["text"],
    labels=train_df["model"],
    max_length=256,
    tokenizer=tokenizer
)

test_datasets = LLMTextData(
    texts=test_df["text"],
    labels=test_df["model"],
    max_length=256,
    tokenizer=tokenizer
)

train_loader = DataLoader(
    dataset=train_datasets,
    batch_size=8,
    shuffle=True,
)

test_loader = DataLoader(
    dataset=test_datasets,
    batch_size=8,
)

print(len(train_loader), len(test_loader))

(2159, 4) (540, 4)
270 68


In [36]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)

cpu


In [52]:
from sklearn.metrics import accuracy_score, recall_score
from tqdm.auto import tqdm

y_true = []
y_pred = []
total_test_batches = len(test_loader)


model.eval()
with torch.no_grad():
    progress_bar = tqdm(test_loader, desc=f"Predicting....")
    for cnt, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        cnt_ += 1
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(pred.cpu().tolist())
        # print(f"Predicting: {cnt+1}/{total_test_batches}")


print("Accuracy:", accuracy_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred, average="macro"))
        


        


Predicting....: 100%|██████████| 68/68 [01:13<00:00,  1.08s/it]

Accuracy: 0.6703703703703704
Recall: 0.6703703703703705


In [48]:
len(y_pred)

68

In [32]:
import torch.nn as nn
from torch.optim import AdamW

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)



In [ ]:
EPOCHS = 1
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_examples = 0
    correct_predictions = 0
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        # print(total_examples, input_ids.shape, attention_mask.shape, labels.shape)
        
        # optimizer -> (clearing gradients)
        optimizer.zero_grad()

        # model output
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # print(outputs.logits.shape)
        logits = outputs.logits
        print(logits)
        # print(torch.argmax(logits, dim=1))
 
        # loss cal
        loss = criterion(outputs.logits, labels)
        # print(loss)

        # backpropogation
        loss.backward()

        # Missing: torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # Clip gradients to prevent explosion and NaN weights
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # update weights
        optimizer.step()

        # total loss calculations
        batch_size = labels.shape[0]
        total_examples += batch_size
        total_loss += loss.item()*batch_size
        predictions = torch.argmax(logits, dim=1)
        correct_predictions += (predictions == labels).sum().item()

    epoch_loss = ( total_loss / total_examples )
    epoch_acc = ( correct_predictions / total_examples )
    print(
        f"Epoch: {epoch + 1} / {EPOCHS} |",
        f"Loss: {epoch_loss:.4f} |",
        f"Accuracy: {epoch_acc:.4f} "
    )
    

tensor([[ 0.3340, -0.0410,  0.0844,  0.0797, -0.0307, -0.2357, -0.0198, -0.0503,
         -0.0693, -0.1711, -0.2029,  0.0480, -0.1367,  0.1812,  0.0665,  0.1908,
         -0.0804, -0.2712],
        [ 0.2566,  0.0710, -0.0321, -0.0126, -0.0947, -0.1241, -0.1389, -0.2097,
         -0.0303, -0.1344, -0.2754,  0.1522,  0.0110,  0.0385,  0.0303,  0.2131,
         -0.1683, -0.2045],
        [ 0.0435, -0.0463,  0.0348, -0.0392, -0.0660, -0.4175, -0.1102,  0.0061,
         -0.1639,  0.0569, -0.1776, -0.0612, -0.0397, -0.0561,  0.0781,  0.2460,
         -0.1210, -0.2371],
        [ 0.2866,  0.0569,  0.0938, -0.0107, -0.1143, -0.2185, -0.1248, -0.1575,
         -0.0867, -0.0484, -0.1069,  0.1332, -0.1569,  0.0764, -0.0010,  0.1125,
          0.0248, -0.2274],
        [ 0.2456, -0.0467,  0.1194,  0.0651, -0.0043,  0.0573, -0.2444, -0.4041,
          0.0261, -0.0904, -0.0371,  0.1389, -0.1171,  0.1389,  0.0013,  0.1006,
          0.1123, -0.0861],
        [ 0.2123,  0.0049,  0.0987,  0.0231, -0.04

KeyboardInterrupt: 

In [ ]:
save_directory = "/kaggle/working/deberta_v3_finetuned"

model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model

model_to_save.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)


from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained(save_directory)
tokenizer = AutoTokenizer.from_pretrained(save_directory)

In [ ]:
import torch
import shutil
import os

save_directory = "/kaggle/working/deberta_v3_finetuned"

# DataParallel wrapper se original Hugging Face model nikalo
model_to_save = (
    model.module
    if isinstance(model, torch.nn.DataParallel)
    else model
)

# Save model weights + config
model_to_save.save_pretrained(save_directory)

# Save tokenizer files
tokenizer.save_pretrained(save_directory)

# Pack the entire directory into a ZIP
zip_path = shutil.make_archive(
    "/kaggle/working/deberta_v3_finetuned",  # ZIP ka output path aur naam
    "zip",                                   #  Archive format ZIP   
    root_dir=save_directory                  #  Kaunse folder ki files ZIP mein daalni hain
)

print("Saved files:", os.listdir(save_directory))
print("ZIP location:", zip_path)

In [ ]:
# for batch in train_loader:
#     # 1. Unpack your data (depending on what your Dataset __getitem__ returns)
#     input_ids = batch['input_ids']
#     attention_mask = batch['attention_mask']
#     labels = batch['labels']
#     print(input_ids.shape, attention_mask.shape, labels.shape)
#     print("*"*50)
#     print()
#     print
#     ()

# Print the total number of batches
# total_batches = len(train_loader)
# print(f"Total number of batches: {total_batches}")


Total number of batches: 338


In [ ]:
# single_batch = next(iter(train_loader))

# # Check the shape of the tensor (e.g., torch.Size([batch_size, max_length]))
# print(single_batch['input_ids'].shape)

torch.Size([8, 181])


In [ ]:
# batch = next(iter(train_loader))
# print("Input IDs", batch["input_ids"].shape)
# print("Attention mask", batch["attention_mask"].shape)
# print("Labels", batch["labels"].shape)

Input IDs torch.Size([8, 181])
Attention mask torch.Size([8, 181])
Labels torch.Size([8])


In [ ]:
# outputs = model(
#     input_ids = batch["input_ids"],
#     attention_mask = batch["attention_mask"]
# )

# print(outputs.logits.shape)

torch.Size([8, 18])


In [ ]:
# from torch.optim import AdamW
# optimizer = AdamW(
#     params=model.parameters(),
#     lr=2e-5  #Learning rate control karta hai ki optimizer parameters ko kitne scale par update kare.
# )

In [ ]:

# from tqdm.auto import tqdm

# EPOCHS = 1

# for epoch in range(EPOCHS):
#     model.train()
#     total_loss = 0.0
#     total_examples = 0
#     correct_predictions = 0

#     progress_bar = tqdm(
#         train_loader,
#         desc=f"Epoch: {epoch + 1}/{EPOCHS}"
#     )

#     for batch in progress_bar:
#         input_ids = batch["input_ids"].to(device)
#         attention_mask = batch["attention_mask"].to(device)
#         labels = batch["labels"].to(device)

#         optimizer.zero_grad()

#         # Forward pass
#         outputs = model(
#             input_ids=input_ids,
#             attention_mask=attention_mask
#         )

#         logits = outputs.logits
#         loss = criterion(logits, labels)

#         # Backpropagation and weight update
#         loss.backward()
#         optimizer.step()

#         # Metrics
#         batch_size = labels.size(0)
#         predictions = torch.argmax(logits, dim=1)

#         correct_predictions += (predictions == labels).sum().item()
#         total_loss += loss.item() * batch_size
#         total_examples += batch_size

#         # Update progress bar after every batch
#         progress_bar.set_postfix(
#             loss=f"{total_loss / total_examples:.4f}",
#             accuracy=f"{correct_predictions / total_examples:.4f}"
#         )

#     epoch_loss = total_loss / total_examples
#     epoch_acc = correct_predictions / total_examples

#     print(
#         f"Epoch {epoch + 1}/{EPOCHS} | "
#         f"Loss: {epoch_loss:.4f} | "
#         f"Accuracy: {epoch_acc:.4f}"
#     )


In [ ]:
# from transformers import AutoTokenizer

# # Point directly to your local folder path
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # Load directly from the disk
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

# text = "Hello how is the microarchitecture of this overcompressive world"
# tokens = tokenizer.tokenize(text)
# print(tokens)


['▁Hello', '▁how', '▁is', '▁the', '▁micro', 'architecture', '▁of', '▁this', '▁over', 'com', 'pressive', '▁world']


In [ ]:
# print(len(tokenizer.tokenize(df.iloc[0]["text"])))

226


In [ ]:
# text = "Ai is a cool tech."
# encoded = tokenizer(text)

# # Tokens 
# print("Tokens:")
# print(encoded.tokens())
# # Input ids
# print("Input Ids:")
# print(encoded["input_ids"])
# # attention mask (A 1 means: "This is a real word/token. Read it and process it.")
# print("Attention mask")
# print(encoded["attention_mask"])
# # tokens from ids
# print("Tokens from Ids:")
# print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))



Tokens:
['[CLS]', '▁Ai', '▁is', '▁a', '▁cool', '▁tech', '.', '[SEP]']
Input Ids:
[1, 21292, 269, 266, 1570, 3539, 260, 2]
Attention mask
[1, 1, 1, 1, 1, 1, 1, 1]
Tokens from Ids:
['[CLS]', '▁Ai', '▁is', '▁a', '▁cool', '▁tech', '.', '[SEP]']


In [ ]:
# # 2. Complete tokenization
# encoded = tokenizer(text)

# print("\nInput IDs:")
# print(encoded["input_ids"])

# print("\nAttention Mask:")
# print(encoded["attention_mask"])

# # 3. IDs ko wapas tokens mein decode karo
# print("\nTokens from IDs:")
# print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

In [ ]:
# import torch

# logits = torch.tensor([[2.0, 1.0, 0.1]])

# # Convert logits to probabilities
# probabilities = torch.softmax(logits, dim=1)

# # Find the class with the highest probability
# predicted_class = torch.argmax(probabilities, dim=1)

# print("Logits:", logits)
# print("Probabilities:", probabilities)
# print("Predicted class:", predicted_class.item())

In [ ]:
# import torch
# print(torch.__version__)
# logits = torch.tensor([ 
#     1.2, -0.4, 2.8, 0.7,
#     0.2, 0.1, -0.8, 0.5,
#     1.0, -0.3, 0.4, 0.6,
#     0.0, 0.9, -0.2, 0.3,
#     0.8, 0.1])

# pred = torch.argmax(logits)
# print(pred.item())



2.14.1+cpu


In [ ]:
# logits = torch.tensor([[2, 1, 0.1]])
# prob = torch.softmax(logits, dim=-1)
# print(torch.argmax(prob))
# # print(pred)

tensor(0)


In [ ]:
# probabilities = torch.softmax(logits, dim=-1)
# predictions = torch.argmax(logits, dim=-1)
# print(probabilities, predictions)

tensor([[0.6590, 0.2424, 0.0986]]) tensor([0])


In [ ]:
# import torch.nn as nn
# logits = torch.tensor([[2, 1, 0.1]])
# label = torch.tensor([1])
# loss_fn = nn.CrossEntropyLoss()
# loss = loss_fn(logits, label)
# print(loss.item())

1.4170299768447876


In [ ]:
# import os
# from transformers import AutoTokenizer

# MODEL_NAME = "microsoft/deberta-v3-small"
# HF_TOKEN = os.getenv("HF_TOKEN")

# # 1. Download from Hugging Face
# tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)

# # 2. Define a folder name on your disk
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # 3. Save it
# tokenizer.save_pretrained(LOCAL_DIR)
# print(f"Tokenizer successfully saved to {LOCAL_DIR}")


In [ ]:
# from transformers import AutoTokenizer

# # Point directly to your local folder path
# LOCAL_DIR = r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\checkpoint"

# # Load directly from the disk
# tokenizer = AutoTokenizer.from_pretrained(LOCAL_DIR)

# text = "Hello how is the microarchitecture of this overcompressive world"
# tokens = tokenizer.tokenize(text)
# print(tokens)
